# SmartQ End-to-End Machine Learning Notebook

This single notebook combines the existing SmartQ machine-learning notebooks in their original order:

1. Data Understanding & Exploratory Data Analysis (EDA)
2. Data Preparation
3. Model Training & Evaluation
4. Model Diagnostics & Statistical Understanding

The original notebook code and explanations are preserved. The only modelling-code change is the correlation block in the EDA section: it uses the full annotated correlation matrix / heatmap requested for the combined notebook.

The existing later correlation section from the original EDA notebook is intentionally left untouched so that no other original notebook content is removed or rewritten.

A Colab repository setup cell is included before the first dataset-loading cell so this notebook can be opened from GitHub and run in Google Colab.


# My SmartQ Data Understanding & Exploratory Data Analysis (EDA)

I use this notebook for the **Data Understanding** stage of my SmartQ CRISP-DM workflow.

I treat EDA as the practical part of Data Understanding, but I know Data Understanding is slightly broader. I use this stage to check data quality, understand my prediction target, identify useful variables, inspect missing values and outliers, and find fields that could cause data leakage.

My prediction target is:

`actual_wait_minutes`

My dataset contains 100,000 synthetic SmartQ operational queue records.

I do this stage first because I do not want to trust model results before I understand the data I am giving the models.


## 1. I load the dataset

I wrote the code below so I can run the notebook either from the repository root or from the `notebooks/` folder.

I want the notebook to be reproducible instead of depending on one exact working directory.


### Colab repository setup

This cell clones the SmartQ repository and moves the Colab runtime into the machine-learning repository snapshot so the original relative dataset paths continue to work.


In [ ]:
!git clone https://github.com/Katlegojack/SmartQ.git
%cd SmartQ/machine_learning/repository_snapshot


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET_NAME = "SmartQ_Synthetic_Operational_Dataset_100k.csv"

candidate_paths = [
    Path("data") / DATASET_NAME,
    Path("..") / "data" / DATASET_NAME,
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Run this notebook from the repository root "
        "or from the notebooks folder."
    )

df = pd.read_csv(data_path)

print("Dataset:", data_path.resolve())
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(df.head())


## 2. I parse timestamps and define my modelling population

For waiting-time regression, I use **completed visits only**.

I exclude cancelled and no-show records because those records do not contain a genuine completed waiting-time outcome.

I keep them in the master dataset because they can still be useful for other SmartQ analysis later.


In [ ]:
datetime_columns = [
    "scenario_date",
    "appointment_at",
    "arrival_at",
    "check_in_at",
    "service_eligible_at",
    "call_time",
    "service_started_at",
    "service_completed_at",
]

for column in datetime_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

completed = df[df["status"] == "COMPLETED"].copy()

print(f"All records: {len(df):,}")
print(f"Completed records available for regression: {len(completed):,}")
print(f"Excluded no-shows/cancellations: {len(df) - len(completed):,}")


## 3. I inspect the basic structure and data types

I check the shape, column types and descriptive statistics before doing deeper analysis.

I do this because wrong data types or unexpected values can create modelling problems later.


In [ ]:
display(df.info())
display(df.describe(include="all").T)


## 4. I inspect status, service, branch and queue composition

I use these counts to understand what kinds of records exist in my dataset and whether the different operational groups are represented.


In [ ]:
for column in [
    "status",
    "branch_name",
    "service_name",
    "booking_source",
    "queue_type",
    "day_of_week",
    "is_peak_period",
]:
    print(f"\n--- {column} ---")
    display(df[column].value_counts(dropna=False).to_frame("count"))


## 5. I analyse missing values

I do not assume every missing value is bad data.

I expect some missing values by design:

- `appointment_at` is blank for walk-ins because they do not have appointments.
- recent-history features can be blank early in the day because there may not yet be enough previous completed/called customers.
- outcome fields are blank for no-shows and cancellations.

My goal is to understand **why** a value is missing before I decide how to handle it.


In [ ]:
missing_all = (
    df.isna()
      .sum()
      .to_frame("missing_rows")
      .assign(missing_pct=lambda x: (x["missing_rows"] / len(df) * 100).round(2))
      .query("missing_rows > 0")
      .sort_values("missing_rows", ascending=False)
)
display(missing_all)

print("\nMissing values inside completed visits only:")
missing_completed = (
    completed.isna()
             .sum()
             .to_frame("missing_rows")
             .assign(missing_pct=lambda x: (x["missing_rows"] / len(completed) * 100).round(2))
             .query("missing_rows > 0")
             .sort_values("missing_rows", ascending=False)
)
display(missing_completed)


## 6. I run data-quality checks

I use these checks to confirm that the synthetic records still obey basic SmartQ queue logic.

I check for impossible waiting times, broken timestamp order, invalid queue positions and service-eligibility problems.

I do this because a model can learn bad data just as easily as good data.


In [ ]:
appointment_rows = completed[completed["booking_source"] == "APPOINTMENT"]
walkin_rows = completed[completed["booking_source"] == "WALK_IN"]

quality_checks = pd.Series({
    "negative_wait_rows": int((completed["actual_wait_minutes"] < 0).sum()),
    "nonpositive_service_rows": int((completed["actual_service_minutes"] <= 0).sum()),
    "queue_position_mismatch": int(
        (completed["queue_position"] != completed["people_ahead"] + 1).sum()
    ),
    "call_before_service_eligibility": int(
        (completed["call_time"] < completed["service_eligible_at"]).sum()
    ),
    "service_start_before_call": int(
        (completed["service_started_at"] < completed["call_time"]).sum()
    ),
    "completion_before_service_start": int(
        (completed["service_completed_at"] < completed["service_started_at"]).sum()
    ),
    "appointment_eligible_before_booked_time": int(
        (appointment_rows["service_eligible_at"] < appointment_rows["appointment_at"]).sum()
    ),
    "walkin_eligibility_not_equal_checkin": int(
        (walkin_rows["service_eligible_at"] != walkin_rows["check_in_at"]).sum()
    ),
}, name="violations")

display(quality_checks.to_frame())


## 7. I understand my target: actual waiting time

I inspect the distribution of `actual_wait_minutes` because this is the number I am trying to predict.

I look at the mean, median, percentiles and histogram so I can see whether the target is balanced, skewed or affected by extreme congestion cases.


In [ ]:
wait_stats = completed["actual_wait_minutes"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)
display(wait_stats.to_frame("actual_wait_minutes"))

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(completed["actual_wait_minutes"], bins=60)
ax.set_title("Distribution of Actual SmartQ Waiting Time")
ax.set_xlabel("Actual wait (minutes)")
ax.set_ylabel("Customers")
plt.show()


## 8. I compare waiting time across service, branch, queue lane and booking source

I use these group summaries to see whether important SmartQ operational groups behave differently before I start modelling.


In [ ]:
group_columns = [
    "service_name",
    "branch_name",
    "queue_type",
    "booking_source",
    "is_peak_period",
]

for column in group_columns:
    summary = (
        completed.groupby(column)["actual_wait_minutes"]
                 .agg(["count", "mean", "median"])
                 .round(2)
                 .sort_values("mean", ascending=False)
    )
    print(f"\n--- Wait by {column} ---")
    display(summary)


In [ ]:
branch_wait = (
    completed.groupby("branch_name")["actual_wait_minutes"]
             .mean()
             .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 5))
branch_wait.plot(kind="barh", ax=ax)
ax.set_title("Average Waiting Time by Branch")
ax.set_xlabel("Average wait (minutes)")
ax.set_ylabel("Branch")
plt.tight_layout()
plt.show()


## 9. I inspect time-of-day and weekday patterns

I check whether waiting time changes by hour and weekday.

I keep these patterns in mind when deciding whether time variables could help the models.


In [ ]:
hourly_wait = completed.groupby("hour_of_day")["actual_wait_minutes"].mean()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
weekday_wait = (
    completed.groupby("day_of_week")["actual_wait_minutes"]
             .mean()
             .reindex(weekday_order)
)

display(hourly_wait.round(2).to_frame("avg_wait_minutes"))
display(weekday_wait.round(2).to_frame("avg_wait_minutes"))

fig, ax = plt.subplots(figsize=(9, 5))
hourly_wait.plot(marker="o", ax=ax)
ax.set_title("Average Waiting Time by Hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Average wait (minutes)")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
weekday_wait.plot(kind="bar", ax=ax)
ax.set_title("Average Waiting Time by Weekday")
ax.set_xlabel("Day")
ax.set_ylabel("Average wait (minutes)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. I compare queue conditions with waiting time

I use this section to check whether queue-state variables move in a sensible direction with waiting time.

I want to know whether features such as people ahead, workload and queue pressure actually contain useful signal before I rely on them in a model.


In [ ]:
import matplotlib.pyplot as plt

numeric_features = [
    "people_ahead",
    "general_waiting",
    "priority_waiting",
    "serving_count",
    "effective_open_counters",
    "counter_utilisation",
    "queue_pressure_index",
    "workload_minutes_ahead",
    "recent_avg_service_minutes_10",
    "recent_avg_wait_minutes_10",
    "recent_throughput_60m",
    "service_target_minutes",
    "hour_of_day",
    "actual_wait_minutes",
]

# Full correlation matrix
corr_matrix = completed[numeric_features].corr(numeric_only=True)

# Show the numeric matrix first
display(corr_matrix.round(2))

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, aspect="auto")

# Axis labels
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_xticklabels(corr_matrix.columns, rotation=90)
ax.set_yticklabels(corr_matrix.index)

# Write correlation values inside each cell
for i in range(len(corr_matrix.index)):
    for j in range(len(corr_matrix.columns)):
        ax.text(
            j,
            i,
            f"{corr_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

# Colorbar + title
fig.colorbar(im, ax=ax, label="Correlation")
ax.set_title("Correlation Heatmap of Smart Q Numeric Features")

plt.tight_layout()
plt.show()


In [ ]:
sample = completed.sample(n=min(5000, len(completed)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    sample["queue_pressure_index"],
    sample["actual_wait_minutes"],
    alpha=0.20,
    s=12,
)
ax.set_title("Queue Pressure vs Actual Waiting Time")
ax.set_xlabel("Queue pressure index")
ax.set_ylabel("Actual wait (minutes)")
plt.show()


## 11. I keep the deterministic ETA as an engineering benchmark

The existing SmartQ ETA is **not one of my three official ML models**.

I keep it as an extra benchmark because I want to know whether the machine-learning models improve on the queue logic SmartQ already has.

I do not use the deterministic ETA as an official model input because I want the three ML models to learn from the queue conditions themselves.


In [ ]:
baseline_error = completed["actual_wait_minutes"] - completed["baseline_eta_minutes"]

baseline_mae = baseline_error.abs().mean()
baseline_rmse = np.sqrt((baseline_error ** 2).mean())

print(f"Deterministic SmartQ ETA MAE:  {baseline_mae:.2f} minutes")
print(f"Deterministic SmartQ ETA RMSE: {baseline_rmse:.2f} minutes")


## 12. I audit for data leakage

For a prediction made at check-in, I only use information that would actually be available at that moment.

I exclude fields that contain future/outcome information or directly reveal the answer.

I do this because data leakage can make a model look unrealistically accurate while making it useless in a real application.


In [ ]:
target = "actual_wait_minutes"

leakage_columns = [
    "call_time",
    "actual_wait_minutes",
    "wait_variance_minutes",
    "actual_service_minutes",
    "service_variance_minutes",
    "counter_number",
    "service_started_at",
    "service_completed_at",
    "status",
    "no_show",
    "service_within_target",
    "wait_within_30_minutes",
]

display(pd.DataFrame({"excluded_leakage_or_outcome_field": leakage_columns}))


## 13. My initial EDA conclusions

Before I move to modelling, I confirm the following:

- I understand the dataset dimensions and status counts.
- I use completed visits as the regression population.
- I understand expected missing values instead of deleting them blindly.
- I found no impossible negative waits or broken timestamp ordering in my defined checks.
- waiting time changes with queue conditions and some branch/time patterns.
- my target is right-skewed, so MAE and RMSE both give me useful information.
- I have excluded leakage fields from model inputs.

My next stage is **Data Preparation**, where I choose the final features, handle missing recent-history values, encode categories and build a chronological train/validation/test split.


## 14. I add clearer visual comparisons

I added these charts because I wanted my EDA to answer practical questions instead of only producing generic graphs.

I ask:

- Do General and Priority customers wait differently?
- Do appointments and walk-ins wait differently?
- Are peak periods worse?
- Do service types have different raw average waits?


In [ ]:
comparison_specs = [
    ("queue_type", "Average wait by queue lane", "Queue lane"),
    ("booking_source", "Average wait by booking source", "Booking source"),
    ("is_peak_period", "Average wait: peak vs non-peak", "Peak period"),
    ("service_name", "Average wait by service type", "Service"),
]

for column, title, xlabel in comparison_specs:
    summary = (
        completed.groupby(column)["actual_wait_minutes"]
                 .mean()
                 .sort_values()
    )
    fig, ax = plt.subplots(figsize=(8, 4.5))
    summary.plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Average wait (minutes)")
    plt.xticks(rotation=25 if column == "service_name" else 0)
    plt.tight_layout()
    plt.show()


## 15. I visualise missing values

A missing value is not automatically bad data.

I use this chart to see **where** missing values occur so I can decide whether the pattern is expected or whether it suggests a data-quality problem.


In [ ]:
missing_completed = completed.isna().sum()
missing_completed = missing_completed[missing_completed > 0].sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
missing_completed.plot(kind="barh", ax=ax)
ax.set_title("Missing values in completed visits")
ax.set_xlabel("Missing rows")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


## 16. I inspect a correlation matrix

**Correlation** tells me how strongly two numeric variables move together.

- Values near +1 mean they tend to rise together.
- Values near -1 mean one tends to fall when the other rises.
- Values near 0 mean little linear relationship.

I use correlation for exploration.

I do **not** treat correlation as proof of causation.


In [ ]:
correlation_columns = [
    "actual_wait_minutes",
    "people_ahead",
    "general_waiting",
    "priority_waiting",
    "serving_count",
    "effective_open_counters",
    "counter_utilisation",
    "queue_pressure_index",
    "workload_minutes_ahead",
    "recent_avg_service_minutes_10",
    "recent_avg_wait_minutes_10",
    "recent_throughput_60m",
]

corr = completed[correlation_columns].corr()

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(correlation_columns)))
ax.set_xticklabels(correlation_columns, rotation=75, ha="right", fontsize=8)
ax.set_yticks(range(len(correlation_columns)))
ax.set_yticklabels(correlation_columns, fontsize=8)
ax.set_title("Correlation matrix: selected SmartQ queue features")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 17. I compare statistical significance with practical importance

Because my dataset is large, very small differences can become statistically significant.

So I report both:

- a **p-value**, which helps me ask whether a difference is statistically detectable;
- an **effect size**, which helps me understand how large the difference actually is.

For two-group comparisons I use Welch's t-test and Cohen's d.

For the three service types I use one-way ANOVA and eta-squared.


In [ ]:
from scipy import stats

def welch_and_d(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)

    t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)

    pooled_sd = np.sqrt(
        ((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1))
        / (len(a)+len(b)-2)
    )
    cohens_d = (a.mean() - b.mean()) / pooled_sd
    return t_stat, p_value, cohens_d

comparisons = []

for label, column, a_value, b_value in [
    ("General vs Priority", "queue_type", "GENERAL", "PRIORITY"),
    ("Appointment vs Walk-in", "booking_source", "APPOINTMENT", "WALK_IN"),
    ("Peak vs Non-peak", "is_peak_period", True, False),
]:
    a = completed.loc[completed[column] == a_value, "actual_wait_minutes"]
    b = completed.loc[completed[column] == b_value, "actual_wait_minutes"]

    t_stat, p_value, d = welch_and_d(a, b)

    comparisons.append({
        "comparison": label,
        "mean_a": a.mean(),
        "mean_b": b.mean(),
        "p_value": p_value,
        "cohens_d": d,
    })

display(pd.DataFrame(comparisons))

service_groups = [
    group["actual_wait_minutes"].to_numpy()
    for _, group in completed.groupby("service_name")
]

f_stat, p_value = stats.f_oneway(*service_groups)

grand_mean = completed["actual_wait_minutes"].mean()
ss_between = sum(
    len(group) * (group.mean() - grand_mean)**2
    for group in service_groups
)
ss_total = ((completed["actual_wait_minutes"] - grand_mean)**2).sum()
eta_squared = ss_between / ss_total

print(f"Service-type ANOVA p-value: {p_value:.4f}")
print(f"Service-type eta-squared: {eta_squared:.8f}")


### What I learned from the significance tests

In my generated data I found statistically detectable differences between:

- General and Priority waits;
- appointments and walk-ins;
- peak and non-peak periods.

However, the effect sizes are small.

I also found that the three service types do **not** have a meaningful raw overall difference in waiting time.

This taught me not to use p-values alone to decide whether a feature matters.


---

# Part 2 — Data Preparation

The following cells are preserved from `02_Data_Preparation.ipynb`.


# My SmartQ Data Preparation

I use this notebook to prepare my SmartQ dataset for waiting-time regression after EDA.

My prediction target is:

`actual_wait_minutes`

I use completed visits only for the regression task because they are the records with a genuine completed waiting-time outcome.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

DATASET_NAME = 'SmartQ_Synthetic_Operational_Dataset_100k.csv'
candidate_paths = [Path('data') / DATASET_NAME, Path('..') / 'data' / DATASET_NAME]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('SmartQ dataset not found.')

df = pd.read_csv(data_path, parse_dates=['scenario_date'])
completed = df[df['status'] == 'COMPLETED'].copy()
print(f'Completed visits: {len(completed):,}')


## 1. I choose my final modelling features

I only use information that is available at prediction/check-in time.

I exclude outcome fields and the existing `baseline_eta_minutes` from the official model inputs.

I do this to prevent data leakage and to keep the model comparison independent from the existing deterministic ETA.


In [ ]:
TARGET = 'actual_wait_minutes'

numeric_features = [
    'arrival_offset_minutes', 'people_ahead', 'general_waiting', 'priority_waiting',
    'serving_count', 'open_general_counters', 'open_priority_counters',
    'effective_open_counters', 'counter_utilisation', 'queue_pressure_index',
    'workload_minutes_ahead', 'recent_avg_service_minutes_10',
    'recent_avg_wait_minutes_10', 'recent_throughput_60m',
    'service_target_minutes', 'hour_of_day'
]

categorical_features = [
    'branch_code', 'service_code', 'booking_source',
    'queue_type', 'day_of_week', 'is_peak_period'
]

features = numeric_features + categorical_features
print(f'Input features: {len(features)}')


## 2. I decide how to handle missing values

Two recent-history features can be missing early in an operating day because there may not yet be enough earlier queue history.

For numeric missing values, I use the **training-set median**.

For categorical missing values, I use the most frequent training category.

I fit these rules using training data only because I do not want validation or test information to influence preprocessing.


## 3. I create a chronological train / validation / test split

I split by **whole operating dates**, so one day never appears in more than one partition.

My date ranges are:

- Training: 2026-01-02 to 2026-06-16
- Validation: 2026-06-17 to 2026-07-22
- Test: 2026-07-23 to 2026-08-27

I chose a chronological split because SmartQ will learn from past queue behaviour and make predictions on later queue behaviour.

I also wanted to prevent customers from the same operating day appearing in both training and test data.


In [ ]:
dates = sorted(completed['scenario_date'].dt.normalize().unique())
n_dates = len(dates)

train_end = pd.Timestamp(dates[int(n_dates * 0.70) - 1])
val_start = pd.Timestamp(dates[int(n_dates * 0.70)])
val_end = pd.Timestamp(dates[int(n_dates * 0.85) - 1])
test_start = pd.Timestamp(dates[int(n_dates * 0.85)])

train = completed[completed['scenario_date'] <= train_end].copy()
validation = completed[(completed['scenario_date'] >= val_start) & (completed['scenario_date'] <= val_end)].copy()
test = completed[completed['scenario_date'] >= test_start].copy()

split_summary = pd.DataFrame({
    'split': ['Train', 'Validation', 'Test'],
    'rows': [len(train), len(validation), len(test)],
    'start': [train['scenario_date'].min().date(), validation['scenario_date'].min().date(), test['scenario_date'].min().date()],
    'end': [train['scenario_date'].max().date(), validation['scenario_date'].max().date(), test['scenario_date'].max().date()],
})
display(split_summary)


### My confirmed split sizes

- **Train:** 64,074 rows (69.15%)
- **Validation:** 14,665 rows (15.83%)
- **Test:** 13,916 rows (15.02%)

My total regression population is **92,655 completed visits**.


In [ ]:
X_train, y_train = train[features], train[TARGET]
X_validation, y_validation = validation[features], validation[TARGET]
X_test, y_test = test[features], test[TARGET]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_validation_prepared = preprocessor.transform(X_validation)
X_test_prepared = preprocessor.transform(X_test)

print('Prepared shapes:')
print('Train:', X_train_prepared.shape)
print('Validation:', X_validation_prepared.shape)
print('Test:', X_test_prepared.shape)


## 4. I protect the model from leakage

I deliberately exclude:

- `call_time`
- `actual_wait_minutes`
- `wait_variance_minutes`
- `actual_service_minutes`
- `service_variance_minutes`
- `counter_number`
- `service_started_at`
- `service_completed_at`
- status-derived outcome flags
- `baseline_eta_minutes`

I exclude these because they either contain future information, contain the target itself, or would make the official ML comparison depend on the existing ETA.

After this stage, my prepared data is ready for the mean baseline, Linear Regression, Random Forest and XGBoost.


---

# Part 3 — Model Training & Evaluation

The following cells are preserved from `03_Model_Training_Evaluation.ipynb`.


# My SmartQ Model Training & Evaluation

I use this notebook to reproduce the official model comparison in my SmartQ proposal.

I train:

- Linear Regression
- Random Forest
- XGBoost

I evaluate them mainly with:

- MAE
- RMSE

I fixed my official selection rule before final test evaluation:

> I select the model with the lowest validation MAE.


## 1. My validation result summary

My official validation results are:

| Model | Validation MAE | Validation RMSE |
|---|---:|---:|
| Linear Regression | 4.1146 | 6.1452 |
| Random Forest | 2.6315 | 4.4920 |
| **XGBoost** | **2.6302** | **4.3893** |

XGBoost has the lowest validation MAE, so I select it according to my declared rule.

Random Forest is extremely close.

I therefore describe the result as a near-tie rather than pretending XGBoost is dramatically better.


## 2. My final unseen test results

After I fixed the model-selection decision using validation data, I evaluated the later test period.

| Model | Test MAE | Test RMSE |
|---|---:|---:|
| Linear Regression | 4.0645 | 6.4606 |
| Random Forest | 2.5231 | 4.6425 |
| **Selected XGBoost** | **2.5824** | **4.9561** |
| Mean baseline | 14.9850 | 24.8143 |
| SmartQ deterministic ETA | 4.6386 | 6.3683 |

Random Forest happens to have a slightly lower test MAE.

I do **not** switch models after seeing that result because the test set is not supposed to select the model.

I keep XGBoost because it won using the validation rule I defined in advance.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Run from repository root or notebooks/.
DATASET_NAME = 'SmartQ_Synthetic_Operational_Dataset_100k.csv'
candidate_paths = [Path('data') / DATASET_NAME, Path('..') / 'data' / DATASET_NAME]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('SmartQ dataset not found.')

df = pd.read_csv(data_path, parse_dates=['scenario_date'])
completed = df[df['status'] == 'COMPLETED'].copy()


In [ ]:
TARGET = 'actual_wait_minutes'
numeric_features = [
    'arrival_offset_minutes','people_ahead','general_waiting','priority_waiting',
    'serving_count','open_general_counters','open_priority_counters','effective_open_counters',
    'counter_utilisation','queue_pressure_index','workload_minutes_ahead',
    'recent_avg_service_minutes_10','recent_avg_wait_minutes_10','recent_throughput_60m',
    'service_target_minutes','hour_of_day'
]
categorical_features = ['branch_code','service_code','booking_source','queue_type','day_of_week','is_peak_period']
features = numeric_features + categorical_features

dates = sorted(completed['scenario_date'].dt.normalize().unique())
n = len(dates)
train_end = pd.Timestamp(dates[int(n*0.70)-1])
val_start = pd.Timestamp(dates[int(n*0.70)])
val_end = pd.Timestamp(dates[int(n*0.85)-1])
test_start = pd.Timestamp(dates[int(n*0.85)])

train = completed[completed['scenario_date'] <= train_end].copy()
validation = completed[(completed['scenario_date'] >= val_start) & (completed['scenario_date'] <= val_end)].copy()
test = completed[completed['scenario_date'] >= test_start].copy()

X_train,y_train = train[features],train[TARGET]
X_val,y_val = validation[features],validation[TARGET]
X_test,y_test = test[features],test[TARGET]

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
])
Xtr = preprocessor.fit_transform(X_train)
Xv = preprocessor.transform(X_val)
Xt = preprocessor.transform(X_test)


## 3. I use baselines

My project requires a simple mean-wait baseline.

I also report the existing deterministic SmartQ ETA as an extra engineering comparison.

I do not use the deterministic ETA as an official model input because I want the ML models to learn directly from the queue conditions.


In [ ]:
def score(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
    }

mean_wait = y_train.mean()
print('Mean baseline:', score(y_val, np.full(len(y_val), mean_wait)))
print('Deterministic ETA:', score(y_val, validation['baseline_eta_minutes']))


## 4. I train Linear Regression

I use Linear Regression as my simple, interpretable reference model.

I want to see how far a straightforward linear relationship can go before I move to more flexible tree models.


In [ ]:
linear = LinearRegression()
linear.fit(Xtr, y_train)
linear_val_pred = np.clip(linear.predict(Xv), 0, None)
print(score(y_val, linear_val_pred))


## 5. I train and lightly tune Random Forest

I compare two deliberately small parameter combinations.

I keep tuning limited because I want the experiment to stay understandable and inside the project scope.

I do not want a huge parameter search to overfit my validation data or turn the project into a tuning competition.


In [ ]:
rf_candidates = [
    {'n_estimators':100,'max_depth':18,'min_samples_leaf':2,'max_features':0.8},
    {'n_estimators':150,'max_depth':14,'min_samples_leaf':2,'max_features':1.0},
]

rf_trials = []
for params in rf_candidates:
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(Xtr, y_train)
    pred = np.clip(model.predict(Xv), 0, None)
    rf_trials.append({**params, **score(y_val, pred)})
display(pd.DataFrame(rf_trials))


## 6. I train and lightly tune XGBoost

I compare a small number of controlled XGBoost configurations.

I use the same training and validation data as the other models so the comparison stays fair.


In [ ]:
xgb_candidates = [
    {'n_estimators':150,'max_depth':4,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9,'reg_lambda':1.0},
    {'n_estimators':200,'max_depth':5,'learning_rate':0.08,'subsample':0.9,'colsample_bytree':0.9,'reg_lambda':2.0},
]

xgb_trials = []
for params in xgb_candidates:
    model = XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror', tree_method='hist', **params)
    model.fit(Xtr, y_train)
    pred = np.clip(model.predict(Xv), 0, None)
    xgb_trials.append({**params, **score(y_val, pred)})
display(pd.DataFrame(xgb_trials))


## 7. I select XGBoost and check traffic scenarios

I select XGBoost using validation MAE.

On my final synthetic test period:

- MAE = **2.5824 minutes**
- RMSE = **4.9561 minutes**

For reporting, I group traffic using `queue_pressure_index`:

- Low: < 1.0
- Moderate: 1.0 to < 2.5
- Busy: >= 2.5

My test MAE increases from about **1.22 minutes in Low traffic** to **7.94 minutes in Busy traffic**.

I treat that as an important limitation rather than hiding it.


## 8. My modelling conclusion

I trained all three required models on the same chronologically separated data and evaluated them with the same MAE/RMSE metrics.

XGBoost achieved the lowest validation MAE, so I keep it as my SmartQ integration candidate.

I do not use the final test set to re-select the winner.

That keeps my evaluation method consistent with the rule I defined before seeing the final answers.


---

# Part 4 — Model Diagnostics & Statistical Understanding

The following cells are preserved from `04_Model_Diagnostics_Statistical_Analysis.ipynb`.


# My SmartQ Model Diagnostics & Statistical Understanding

I use this notebook to go beyond MAE and RMSE.

I do not want my ML work to become a black box that I cannot explain.

I ask:

- Is each model a good fit?
- Is there evidence of overfitting?
- What does R² tell me?
- Which Linear Regression terms are statistically significant?
- Are some predictors repeating the same information?
- Do the main Linear Regression assumptions hold?
- Which variables matter most to XGBoost?
- What do permutation importance and SHAP tell me?
- Where does the selected model still fail?

My most important lesson here is:

> Statistical significance is not the same as practical importance or predictive usefulness.


## 1. I load the same SmartQ data and recreate the chronological split

I reuse the exact modelling population and feature set from my official experiment.

I do this because diagnostics should explain the same models I evaluated earlier, not a different experiment.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera, durbin_watson
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from xgboost import XGBRegressor, DMatrix

DATASET_NAME = "SmartQ_Synthetic_Operational_Dataset_100k.csv"
candidate_paths = [Path("data") / DATASET_NAME, Path("..") / "data" / DATASET_NAME]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("SmartQ dataset not found.")

df = pd.read_csv(data_path, parse_dates=["scenario_date"])
completed = df[df["status"] == "COMPLETED"].copy()

TARGET = "actual_wait_minutes"
numeric_features = [
    "arrival_offset_minutes","people_ahead","general_waiting","priority_waiting",
    "serving_count","open_general_counters","open_priority_counters",
    "effective_open_counters","counter_utilisation","queue_pressure_index",
    "workload_minutes_ahead","recent_avg_service_minutes_10",
    "recent_avg_wait_minutes_10","recent_throughput_60m",
    "service_target_minutes","hour_of_day",
]
categorical_features = [
    "branch_code","service_code","booking_source",
    "queue_type","day_of_week","is_peak_period",
]
features = numeric_features + categorical_features

dates = sorted(completed["scenario_date"].dt.normalize().unique())
n = len(dates)
train_end = pd.Timestamp(dates[int(n * 0.70) - 1])
val_start = pd.Timestamp(dates[int(n * 0.70)])
val_end = pd.Timestamp(dates[int(n * 0.85) - 1])
test_start = pd.Timestamp(dates[int(n * 0.85)])

train = completed[completed["scenario_date"] <= train_end].copy()
validation = completed[
    (completed["scenario_date"] >= val_start)
    & (completed["scenario_date"] <= val_end)
].copy()
test = completed[completed["scenario_date"] >= test_start].copy()

print(len(train), len(validation), len(test))


## 2. I fit the same preprocessing and official models

I repeat the official model settings so my diagnostics are performed on the same experiment.

I want the diagnostic conclusions to match the models I actually compared.


In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

X_train, y_train = train[features], train[TARGET].to_numpy()
X_val, y_val = validation[features], validation[TARGET].to_numpy()
X_test, y_test = test[features], test[TARGET].to_numpy()

Xtr = preprocessor.fit_transform(X_train)
Xv = preprocessor.transform(X_val)
Xt = preprocessor.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, max_depth=14, min_samples_leaf=2,
        max_features=1.0, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
        random_state=42, n_jobs=-1, objective="reg:squarederror",
        tree_method="hist"
    ),
}

for model in models.values():
    model.fit(Xtr, y_train)


## 3. I check good fit and generalisation using MAE, RMSE and R²

**R² (R-squared)** tells me how much of the variation in waiting time the model explains.

A high training R² alone is not enough.

I also want validation and test performance to stay strong, because that tells me whether the model generalises beyond the data it learned from.


In [ ]:
def score(y_true, raw_pred):
    pred = np.clip(np.asarray(raw_pred), 0, None)
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": mean_squared_error(y_true, pred) ** 0.5,
        "R2": r2_score(y_true, pred),
    }

rows = []
for name, model in models.items():
    for split_name, X, y in [
        ("train", Xtr, y_train),
        ("validation", Xv, y_val),
        ("test", Xt, y_test),
    ]:
        rows.append({"model": name, "split": split_name, **score(y, model.predict(X))})

fit_table = pd.DataFrame(rows)
display(fit_table)


In [ ]:
pivot = fit_table.pivot(index="model", columns="split", values="MAE")[["train","validation","test"]]
ax = pivot.plot(kind="bar", figsize=(9,5))
ax.set_title("MAE by model and data split")
ax.set_ylabel("MAE (minutes)")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### What I learn from the fit comparison

Random Forest fits my training data most aggressively because its training MAE is much lower than its validation MAE.

XGBoost has a smaller train-to-validation gap, so its behaviour looks more stable in this experiment.

I do not see catastrophic overfitting because validation and test performance remain close.


## 4. I test statistical significance for Linear Regression

Classic p-values make the most sense for a statistical Linear Regression model.

For inference, I fit a companion OLS model using reference-category encoding and **HC3 robust standard errors**.

I use HC3 because I found that the residual variance is not constant across queue conditions.


In [ ]:
train_ols = train.copy()
for col in numeric_features:
    if train_ols[col].isna().any():
        train_ols[col] = train_ols[col].fillna(train_ols[col].median())

formula = TARGET + " ~ " + " + ".join(
    numeric_features + [f"C({c})" for c in categorical_features]
)

ols = smf.ols(formula, data=train_ols).fit(cov_type="HC3")

coef_table = pd.DataFrame({
    "coefficient": ols.params,
    "std_error_HC3": ols.bse,
    "p_value": ols.pvalues,
    "ci_low": ols.conf_int()[0],
    "ci_high": ols.conf_int()[1],
})
coef_table["significant_0_05"] = coef_table["p_value"] < 0.05

print(f"R²: {ols.rsquared:.4f}")
print(f"Adjusted R²: {ols.rsquared_adj:.4f}")
display(coef_table.sort_values("p_value", ascending=False))


### How I understand a p-value

A p-value asks, roughly:

> If there were really no coefficient effect, how surprising would my result be?

A common threshold is p < 0.05.

Because I have more than 64,000 training rows, even tiny effects can become statistically significant.

So I do **not** automatically treat a small p-value as proof that a feature is important for prediction.


## 5. I compare group significance and effect sizes

I compare a few simple EDA groups.

I use **Cohen's d** because it helps me separate:

- statistically detectable differences;
- differences that are actually large in standardised terms.

I do this because a huge dataset can make tiny differences look statistically impressive.


In [ ]:
def welch_and_d(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    t, p = stats.ttest_ind(a, b, equal_var=False)
    pooled_sd = np.sqrt(
        ((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1))
        / (len(a)+len(b)-2)
    )
    d = (a.mean() - b.mean()) / pooled_sd
    return t, p, d

comparisons = []
for label, column, a_value, b_value in [
    ("General vs Priority", "queue_type", "GENERAL", "PRIORITY"),
    ("Appointment vs Walk-in", "booking_source", "APPOINTMENT", "WALK_IN"),
    ("Peak vs Non-peak", "is_peak_period", True, False),
]:
    a = completed.loc[completed[column] == a_value, TARGET]
    b = completed.loc[completed[column] == b_value, TARGET]
    t, p, d = welch_and_d(a, b)
    comparisons.append({
        "comparison": label,
        "mean_a": a.mean(),
        "mean_b": b.mean(),
        "p_value": p,
        "cohens_d": d,
    })

display(pd.DataFrame(comparisons))

service_groups = [
    group[TARGET].to_numpy()
    for _, group in completed.groupby("service_name")
]
f_stat, p_value = stats.f_oneway(*service_groups)

grand_mean = completed[TARGET].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in service_groups)
ss_total = ((completed[TARGET] - grand_mean)**2).sum()
eta_squared = ss_between / ss_total

print("Service ANOVA F:", f_stat)
print("Service ANOVA p:", p_value)
print("Service eta-squared:", eta_squared)


## 6. I check multicollinearity using VIF

**Multicollinearity** means several predictors repeat similar information.

**VIF (Variance Inflation Factor)** helps me measure that overlap.

I care about high VIF mainly when I interpret individual Linear Regression coefficients.


In [ ]:
numeric_train = train[numeric_features].copy()
numeric_train = numeric_train.fillna(numeric_train.median(numeric_only=True))

vif_input = sm.add_constant(numeric_train.astype(float), has_constant="add")
vif_rows = []
for i, column in enumerate(vif_input.columns):
    if column == "const":
        continue
    vif_rows.append({
        "feature": column,
        "VIF": variance_inflation_factor(vif_input.values, i),
    })

vif_table = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
display(vif_table)

top = vif_table.head(10).sort_values("VIF")
ax = top.plot(kind="barh", x="feature", y="VIF", legend=False, figsize=(9,5))
ax.axvline(10, linestyle="--")
ax.set_title("VIF: strongest overlapping numeric predictors")
ax.set_xlabel("VIF")
plt.tight_layout()
plt.show()


### What I found in SmartQ

Queue pressure, people ahead, serving count, counter utilisation, general waiting and workload ahead have high VIF.

That makes sense because they describe related parts of the same queue state.

I keep them in my tree models because prediction is my main goal.

I do **not** treat every Linear Regression coefficient as a clean independent causal effect.


## 7. I test Linear Regression assumptions

I use formal tests to understand whether the classical OLS assumptions fit my residual behaviour.

I do this to learn where standard statistical interpretation needs extra caution.


In [ ]:
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(ols.resid, ols.model.exog)
jb_stat, jb_p, jb_skew, jb_kurtosis = jarque_bera(ols.resid)
dw = durbin_watson(ols.resid)

assumptions = pd.DataFrame([
    ["Breusch-Pagan LM", bp_lm, bp_lm_p],
    ["Breusch-Pagan F", bp_f, bp_f_p],
    ["Jarque-Bera", jb_stat, jb_p],
    ["Durbin-Watson", dw, np.nan],
], columns=["diagnostic","statistic","p_value"])

display(assumptions)
print("Residual skewness:", jb_skew)
print("Residual kurtosis:", jb_kurtosis)


The Breusch-Pagan result shows **heteroscedasticity**, which means my error size changes across queue conditions.

The Jarque-Bera result shows my residuals are not normally distributed and have heavy tails.

Because of that, I use robust HC3 standard errors rather than pretending the classical assumptions are perfect.


## 8. I inspect residuals and negative raw predictions

A **residual** is:

actual wait - predicted wait

If the residual is positive, my model predicted too low.

If the residual is negative, my model predicted too high.

I use residuals to understand the shape and bias of model errors instead of only looking at one average score.


In [ ]:
residual_rows = []
for name, model in models.items():
    for split_name, X, y in [
        ("train", Xtr, y_train),
        ("validation", Xv, y_val),
        ("test", Xt, y_test),
    ]:
        raw = model.predict(X)
        pred = np.clip(raw, 0, None)
        residual = y - pred
        residual_rows.append({
            "model": name,
            "split": split_name,
            "mean_residual": residual.mean(),
            "median_residual": np.median(residual),
            "negative_raw_predictions": int((raw < 0).sum()),
        })

display(pd.DataFrame(residual_rows))

xgb_test_raw = models["XGBoost"].predict(Xt)
xgb_test_pred = np.clip(xgb_test_raw, 0, None)
xgb_residual = y_test - xgb_test_pred

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(xgb_residual, bins=70)
ax.set_title("XGBoost test residual distribution")
ax.set_xlabel("Residual = actual - predicted (minutes)")
ax.set_ylabel("Customers")
plt.show()


My XGBoost model produces some small negative raw predictions.

Waiting time cannot be negative in the real world, so my customer-facing prediction helper applies:

`max(0, prediction)`

I keep this rule explicit because it is a real-world constraint, not something I want hidden inside the model.


## 9. I calculate permutation importance

Permutation importance asks:

> If I destroy one feature's information by shuffling it, how much worse does my model become?

A large increase in MAE tells me the feature carried useful predictive information.


In [ ]:
xgb_model = models["XGBoost"]
baseline_mae = mean_absolute_error(y_test, xgb_test_pred)

rng = np.random.default_rng(42)
permutation_rows = []
X_test_raw = test[features].copy()

for feature in features:
    shuffled = X_test_raw.copy()
    values = shuffled[feature].to_numpy(copy=True)
    rng.shuffle(values)
    shuffled[feature] = values

    pred = np.clip(
        xgb_model.predict(preprocessor.transform(shuffled)),
        0,
        None,
    )
    mae = mean_absolute_error(y_test, pred)
    permutation_rows.append({
        "feature": feature,
        "permuted_mae": mae,
        "mae_increase": mae - baseline_mae,
    })

permutation_table = pd.DataFrame(permutation_rows).sort_values(
    "mae_increase", ascending=False
)
display(permutation_table)

top = permutation_table.head(10).sort_values("mae_increase")
ax = top.plot(
    kind="barh", x="feature", y="mae_increase",
    legend=False, figsize=(9,5)
)
ax.set_title("Permutation importance")
ax.set_xlabel("Increase in test MAE after shuffling (minutes)")
plt.tight_layout()
plt.show()


## 10. I use SHAP / TreeSHAP

**SHAP** estimates how much each feature pushes a model prediction up or down.

For learning, I use XGBoost's built-in TreeSHAP contributions on a fixed 5,000-row test sample.

I use this to understand model behaviour, not to claim that the features are causal.


In [ ]:
feature_names = preprocessor.get_feature_names_out()
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(Xt), size=min(5000, len(Xt)), replace=False)

dmatrix = DMatrix(Xt[sample_idx], feature_names=list(feature_names))
contributions = xgb_model.get_booster().predict(
    dmatrix, pred_contribs=True
)

shap_table = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(contributions[:, :-1]).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

display(shap_table.head(15))


## 11. My final diagnostic conclusion

I keep XGBoost as my integration candidate.

I do not change the model after diagnostics because:

- validation/test performance remains strong;
- I do not see a catastrophic generalisation collapse;
- the strongest features make operational sense;
- permutation importance and SHAP agree on the main queue drivers;
- the statistical problems mostly affect how I interpret Linear Regression coefficients rather than invalidating the tree-model comparison.

What I learned:

- Random Forest fits training data more aggressively.
- XGBoost generalises more evenly in this experiment.
- several engineered queue variables are highly correlated.
- p-values alone are not enough to judge usefulness.
- classical OLS assumptions are not perfect.
- robust inference is more appropriate.
- workload ahead, people ahead and counter capacity are major predictive signals.
- busy queues remain the hardest operating condition.
- non-negative clipping is necessary for customer-facing regression predictions.

The diagnostics did not give me a new winner.

They gave me a better understanding of the model I already selected.
